In [22]:
import numpy as np

In [23]:
# state will be the 1d board with 0s for empty, 1s for Xs and 2s for Os

In [ ]:
class Board:
    def __init__(self, board_size=(3,3)):

        self.rows, self.cols = board_size
        self.board_size = self.rows * self.cols # 1D board size

        self.board = np.zeros((self.rows*self.cols)) # 1 for Os, 2 for Xs

        self.powers = 3**np.arange(9)

        self.win_lines = [[0,1,2], # top row
                          [3,4,5], # mid row
                          [6,7,8], # bot row
                          [0,3,6], # 1st col
                          [1,4,7], # 2nd col
                          [2,5,8], # 3rd col
                          [0,4,8], # main diag
                          [2,4,6]  # other diag
                          ]
    
    
    def reset(self):
        self.board = np.zeros((self.board_size))
        return self.board

    # return unique int for every state
    def get_state_key(self, state):
        return state @ self.powers
    
    def step(self, action, player):
        
        self.board[action] = player     # update board with new move
        done, winner = self.is_terminal(self.board)   # check if game ended

        if done and winner==player:
            return self.board, 1, done  # this player won
        elif done and winner!=player:
            return self.board, -1, done  # opponent won
        else:
            return self.board, 0, done  # game hasnt ended or draw
        
    def is_terminal(self, board):

        # bool arrays (1 or 0)
        board1 = (board == 1)
        board2 = (board == 2)

        # defaults if not done
        done = False
        winner = 0
        
        for line in self.win_lines:
            if board1[line].all():
                winner = 1
                done = True
                break

            if board2[line].all():
                winner = 2
                done = True
                break
        
        # draw check must be afterwards (if no 3s found AND full board) -> could win on final move
        if not done and sum(board==0) == 0:   
            winner = 0
            done = True

        return done, winner
        

In [ ]:
class MCAgent:
    def __init__(self, board_size, gamma=0.9, alpha=0.1, epsilon=1.0, epsilon_decay=0.9999, min_epsilon=0.0):

        self.board_size = board_size
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

        self.powers = 3**np.arange(board_size)   # [1,3,9,27,81...]
        self.q_table = {}

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)
 
    def get_q_values(self, state):

        state_key = self.get_state_key(state)

        if state_key not in self.q_table:
            self.q_table[state_key] = np.zeros(self.board_size)

        return self.q_table[state_key]

    # return unique int for every state
    def get_state_key(self, state):
        return state @ self.powers

    def get_action(self, state):  # state is 1D arr 

        # Epsilon Greedy
        valid_actions = (state == 0)    # bool mask [1,0,0,1...]

        # Exploration - return random action
        if np.random.uniform(0,1) < self.epsilon:
            action = np.random.choice(np.flatnonzero(valid_actions))    # indices where valid_actions!=0 (board==0) -> rand idx
            return action
        
        # Exploitation
        masked_q_values = np.where(valid_actions, self.get_q_values(state), -np.inf)
        best_action = np.argmax(masked_q_values)

        return best_action


    def update(self, episode=None):

        # episode -> [(s,a,r)...]

        # first visit MC
        states, actions, rewards = zip(*episode) 

        # compute return
        # G_t = R + γ * G_t+1
        # G_1 = R₁ + γR₂ + γ²R₃ + … + γⁿ⁻¹Rₙ
        G = 0
        returns = []
        for R in reversed(rewards):
            G = R + self.gamma * G
            returns.append(G)
        returns.reverse()

        visited_pairs = set()
        
        for (state, action, G) in zip(states, actions, returns):

            state_key = self.get_state_key(state)
            sa_pair = (state_key,action)    # cant use state since not hashable

            if sa_pair not in visited_pairs:

                visited_pairs.add(sa_pair)

                q_vals = self.get_q_values(state)
                old_q = q_vals[action]

                self.q_table[state_key][action] = old_q + self.alpha * (G - old_q)




In [ ]:
# TO DO
class TDAgent:
    def __init__(self, board_size, method=None, gamma=0.9, alpha=0.1, epsilon=1.0, epsilon_decay=0.9999, min_epsilon=0.0):

        self.board_size = board_size   # 1d
        self.method = method
        self.gamma = gamma
        self.alpha = alpha
        self.epsilon = epsilon
        self.epsilon_decay = epsilon_decay
        self.min_epsilon = min_epsilon

        self.powers = 3**np.arange(board_size)   # [1,3,9,27,81...]
        self.q_table = {}

    def decay_epsilon(self):
        self.epsilon = max(self.min_epsilon, self.epsilon * self.epsilon_decay)
 
    def get_q_values(self, state):

        state_key = self.get_state_key(state)

        if state_key not in self.q_table:
            self.q_table[state_key] = np.zeros(self.board_size)

        return self.q_table[state_key]

    # return unique int for every state
    def get_state_key(self, state):
        return state @ self.powers

    def get_action(self, state):  # state is 1D arr 

        # Epsilon Greedy
        valid_actions = (state == 0)    # bool mask [1,0,0,1...]

        # Exploration - return random action
        if np.random.uniform(0,1) < self.epsilon:
            action = np.random.choice(np.flatnonzero(valid_actions))    # indices where valid_actions!=0 (board==0) -> rand idx
            return action
        
        # Exploitation
        masked_q_values = np.where(valid_actions, self.get_q_values(state), -np.inf)
        best_action = np.argmax(masked_q_values)

        return best_action


    def update(self, SARSA=None, done=None):
    
        state, action, R, next_state, next_action = SARSA

        q_vals = self.get_q_values(state)
        old_q = q_vals[action]

        new_q_vals = self.get_q_values(next_state)

        if not done:
            if self.method == "sarsa":
                new_q = new_q_vals[next_action]
            elif self.method == "qlearning":
                new_q = np.max(new_q_vals)
        else:
            new_q = 0.0
            
        G = R + self.gamma * new_q

        state_key = self.get_state_key(state)
        self.q_table[state_key][action] = old_q + self.alpha * (G - old_q)
        


In [ ]:
# Main for MC

env = Board()
agent = MCAgent(env.board_size)  # qlearning, sarsa     # pass 1d board size

episodes = 50000

for e in range(episodes):
    state = env.reset()
    
    done = False
    player = 1  # switches to 2

    episode_memory1 = []
    episode_memory2 = []


    while not done:
        # player1
        state_before1 = state.copy()
        action1 = agent.get_action(state)
        state, reward1, done = env.step(action1, player=1)

        if done:
            reward2 = -reward1  # opposite reward for other agent
            episode_memory1.append((state_before1, action1, reward1))

            # give p2 final reward for their last move (change reward of last move)
            episode_memory2[-1] = (episode_memory2[-1][0], episode_memory2[-1][1], reward2)
            break

        # player2
        state_before2 = state.copy()
        action2 = agent.get_action(state)
        state, reward2, done = env.step(action2, player=2)
        
        if done:
            reward1 = -reward2  # opposite reward for other agent
            episode_memory2.append((state_before2, action2, reward2))

            # give p1 final reward for their last move (change reward of last move)
            episode_memory1[-1] = (episode_memory1[-1][0], episode_memory1[-1][1], reward1)
            break

        # else no one won yet
        reward1 = 0
        reward2 = 0

        episode_memory1.append((state_before1, action1, reward1))
        episode_memory2.append((state_before2, action2, reward2))


    agent.update(episode=episode_memory1)
    agent.update(episode=episode_memory2)
    
    print(f"Episode: {e}    eps: {agent.epsilon}")
    
    # decay epsilon once per episode
    agent.decay_epsilon()

    

Episode: 0    eps: 1.0
Episode: 1    eps: 0.9999
Episode: 2    eps: 0.9998000100000001
Episode: 3    eps: 0.9997000299990001
Episode: 4    eps: 0.9996000599960002
Episode: 5    eps: 0.9995000999900007
Episode: 6    eps: 0.9994001499800017
Episode: 7    eps: 0.9993002099650037
Episode: 8    eps: 0.9992002799440072
Episode: 9    eps: 0.9991003599160128
Episode: 10    eps: 0.9990004498800211
Episode: 11    eps: 0.9989005498350332
Episode: 12    eps: 0.9988006597800497
Episode: 13    eps: 0.9987007797140718
Episode: 14    eps: 0.9986009096361004
Episode: 15    eps: 0.9985010495451367
Episode: 16    eps: 0.9984011994401822
Episode: 17    eps: 0.9983013593202382
Episode: 18    eps: 0.9982015291843062
Episode: 19    eps: 0.9981017090313877
Episode: 20    eps: 0.9980018988604846
Episode: 21    eps: 0.9979020986705985
Episode: 22    eps: 0.9978023084607315
Episode: 23    eps: 0.9977025282298854
Episode: 24    eps: 0.9976027579770624
Episode: 25    eps: 0.9975029977012647
Episode: 26    eps: 0.9

In [ ]:
# Main for TD - TO DO ################

env = Board()
agent = MCAgent()  # qlearning, sarsa

episodes = 10000

for e in range(episodes):
    state = env.reset()
    
    done = False
    player = 1  # switches to 2

    if agent.method == "mc":
        episode_memory1 = []
        episode_memory2 = []

    prev_state1 = None
    prev_action1 = None
    prev_state2 = None
    prev_action2 = None

    while not done:
        # player1
        state_before1 = state.copy()
        action1 = agent.get_action(state)
        state, reward1, done = env.step(action1, player=1)

        if done:
            reward2 = -reward1  # opposite reward for other agent
            if agent.method == "mc":
                episode_memory1.append((state_before1, action1, reward1))

                # give p2 final reward for their last move (change reward of last move)
                episode_memory2[-1] = (episode_memory2[-1][0], episode_memory2[-1][1], reward2)
            break

        # player2
        state_before2 = state.copy()
        action2 = agent.get_action(state)
        state, reward2, done = env.step(action2, player=2)
        
        if done:
            reward1 = -reward2  # opposite reward for other agent
            if agent.method == "mc":
                episode_memory2.append((state_before2, action2, reward2))

                # give p1 final reward for their last move (change reward of last move)
                episode_memory1[-1] = (episode_memory1[-1][0], episode_memory1[-1][1], reward1)
            break

        # else no one won yet
        reward1 = 0
        reward2 = 0

        episode_memory1.append((state_before1, action1, reward1))
        episode_memory2.append((state_before2, action2, reward2))

        #if agent.method != "mc":
            #agent.update(SARSA=(state,action,reward,next_state,next_action), done=done)

    if agent.method == "mc":
        agent.update(episode=episode_memory1)
        agent.update(episode=episode_memory2)
        
    # decay epsilon once per episode
    agent.decay_epsilon()

In [36]:
# helper

def print_board(board, board_size):

    BOARD = [" ", "X", "O"]
    rows, cols = board_size

    #for i in range(rows):
        #print(i, end=" ")
    #print()

    for i in range(rows):
        for j in range(cols):
            print(f"|{BOARD[board[i,j]]}", end="")
        print("|")
        print("-------")
        
    print()

In [49]:
# MC eval

board = np.zeros(env.board.size, dtype=np.int8)
rows, cols = env.rows, env.cols

while True:
    # player1 - you
    valid = False
    row = -1
    col = -1
    while valid==False:
        while row<0 or row>=rows:
            row = int(input("enter row: "))
        while col<0 or col>=cols:
            col = int(input("enter col: "))

        flat_idx = row * cols + col
        if board[flat_idx] == 0:
            board[flat_idx] = 1
            valid = True

    print_board(board.reshape(rows,cols), board_size=(rows,cols))
    
    done, winner = env.is_terminal(board)
    if done and winner!=0:
        print(f"player {winner} won")
        break
    elif done and winner==0:
        print("draw")
        break

    # player2
    state_key = agent.get_state_key(state=board)
    valid = (board==0)
    q_vals = agent.q_table[state_key]
    masked_q_vals = np.where(valid, q_vals, -np.inf)
    action = np.argmax(masked_q_vals)    # take best action
    board[action] = 2

    print_board(board.reshape(env.rows,env.cols), board_size=(env.rows,env.cols))

    done, winner = env.is_terminal(board)
    if done and winner!=0:
        print(f"player {winner} won")
        break
    elif done and winner==0:
        print("draw")
        break

    

| | | |
-------
| | | |
-------
| | |X|
-------

| | | |
-------
| |O| |
-------
| | |X|
-------

| |X| |
-------
| |O| |
-------
| | |X|
-------

| |X|O|
-------
| |O| |
-------
| | |X|
-------

| |X|O|
-------
| |O| |
-------
|X| |X|
-------

| |X|O|
-------
| |O| |
-------
|X|O|X|
-------

|X|X|O|
-------
| |O| |
-------
|X|O|X|
-------

|X|X|O|
-------
|O|O| |
-------
|X|O|X|
-------

|X|X|O|
-------
|O|O|X|
-------
|X|O|X|
-------

draw


In [53]:
print(agent.q_table[6726],"\n")
print(agent.q_table[6723],"\n")
print(agent.q_table[6561],"\n")

[-0.0225909   0.          0.88019008  0.39052706  0.          0.18376574
 -0.22619939  0.06303488  0.        ] 

[-0.96344763 -0.96467451 -0.96236861 -0.96560884  0.         -0.96317193
 -0.96268332 -0.96517214  0.        ] 

[-0.39635132 -0.2280343  -0.68807908 -0.35985699  0.78047954 -0.75084697
 -0.62899679 -0.59524684  0.        ] 

